<a href="https://colab.research.google.com/github/jamesalv/HateXplain-FinalProject/blob/main/mainv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import yaml
import os
from tqdm import tqdm
from collections import Counter
from transformers import AutoTokenizer
import numpy as np
import torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Get the project root directory (assuming notebook is in notebooks/ subfolder)
current_dir = os.getcwd()
base_path = os.path.join(current_dir, 'drive', 'MyDrive', 'Thesis')

In [4]:
config_path = os.path.join(base_path, 'configs', 'base.yaml')
data_path = os.path.join(base_path, 'data', 'dataset.json')

In [5]:
#For de-obsfucating profanities
!git clone https://github.com/dnozza/profanity-obfuscation.git

Cloning into 'profanity-obfuscation'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 101 (delta 48), reused 54 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 24.36 KiB | 4.87 MiB/s, done.
Resolving deltas: 100% (48/48), done.


# Preprocessing

## Raw Data Preprocessing

In [7]:
import re
import string

def deobfuscate_text(text):
    """
    Normalize common text obfuscation patterns to reveal original words.
    Useful for hate speech detection and content analysis.

    Args:
        text (str): Input text with potential obfuscations

    Returns:
        str: Text with obfuscations normalized
    """
    if not isinstance(text, str):
        return text

    # Make a copy to work with
    result = text.lower()

    # 1. Handle asterisk/symbol replacements
    symbol_patterns = {
        # Common profanity
        r'f\*+c?k': 'fuck',
        r'f\*+': 'fuck',
        r's\*+t': 'shit',
        r'b\*+ch': 'bitch',
        r'a\*+s': 'ass',
        r'd\*+n': 'damn',
        r'h\*+l': 'hell',
        r'c\*+p': 'crap',

        # Slurs and hate speech terms (be comprehensive for detection)
        r'n\*+g+[aer]+': 'nigger',  # Various n-word obfuscations
        r'f\*+g+[ot]*': 'faggot',
        r'r\*+[dt]ard': 'retard',
        r'sp\*+c': 'spic',

        # Other symbols
        r'@ss': 'ass',
        r'b@tch': 'bitch',
        r'sh!t': 'shit',
        r'f#ck': 'fuck',
        r'd@mn': 'damn',
    }

    for pattern, replacement in symbol_patterns.items():
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)

    # 2. Handle character spacing (f u c k -> fuck)
    spacing_patterns = {
        r'\bf\s+u\s+c\s+k\b': 'fuck',
        r'\bs\s+h\s+i\s+t\b': 'shit',
        r'\bd\s+a\s+m\s+n\b': 'damn',
        r'\bh\s+e\s+l\s+l\b': 'hell',
        r'\ba\s+s\s+s\b': 'ass',
        r'\bc\s+r\s+a\s+p\b': 'crap',
    }

    for pattern, replacement in spacing_patterns.items():
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)

    # 3. Handle number/letter substitutions
    leet_patterns = {
        # Basic leet speak
        r'\b3\s*1\s*1\s*3\b': 'elle',  # 3113 -> elle
        r'\bf4g\b': 'fag',
        r'\bf4gg0t\b': 'faggot',
        r'\bn00b\b': 'noob',
        r'\bl33t\b': 'leet',
        r'\bh4t3\b': 'hate',
        r'\b5h1t\b': 'shit',
        r'\bf0ck\b': 'fock',

        # Number substitutions
        r'(\w*)0(\w*)': r'\1o\2',  # 0 -> o
        r'(\w*)1(\w*)': r'\1i\2',  # 1 -> i
        r'(\w*)3(\w*)': r'\1e\2',  # 3 -> e
        r'(\w*)4(\w*)': r'\1a\2',  # 4 -> a
        r'(\w*)5(\w*)': r'\1s\2',  # 5 -> s
        r'(\w*)7(\w*)': r'\1t\2',  # 7 -> t
    }

    for pattern, replacement in leet_patterns.items():
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)

    # 4. Handle repeated characters and separators
    # Remove excessive punctuation between letters
    result = re.sub(r'([a-z])[^\w\s]+([a-z])', r'\1\2', result)

    # Handle underscore separation
    result = re.sub(r'([a-z])_+([a-z])', r'\1\2', result)

    # Handle dot separation
    result = re.sub(r'([a-z])\.+([a-z])', r'\1\2', result)

    # 5. Handle common misspellings/variations used for evasion
    evasion_patterns = {
        r'\bfuk\b': 'fuck',
        r'\bfuq\b': 'fuck',
        r'\bfck\b': 'fuck',
        r'\bshyt\b': 'shit',
        r'\bshit\b': 'shit',
        r'\bbiatch\b': 'bitch',
        r'\bbeatch\b': 'bitch',
        r'\basshole\b': 'asshole',
        r'\ba55hole\b': 'asshole',
        r'\btard\b': 'retard',
        r'\bfagg\b': 'fag',
    }

    for pattern, replacement in evasion_patterns.items():
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)

    # 6. Clean up multiple spaces
    result = re.sub(r'\s+', ' ', result).strip()

    return result

# Test function
# def test_deobfuscation():
#     """Test the deobfuscation function with various patterns"""
#     test_cases = [
#         "f*ck this sh*t",
#         "what the f**k",
#         "f u c k you",
#         "this is bull5h1t",
#         "st*pid a**hole",
#         "f.u.c.k that",
#         "fu_ck_ing hell",
#         "what a r*tard",
#         "f@cking idiot",
#         "go to h*ll",
#         "piece of cr@p",
#         "fuk this",
#         "that's fuked up"
#     ]

#     print("Deobfuscation Test Results:")
#     print("-" * 40)
#     for test in test_cases:
#         cleaned = deobfuscate_text(test)
#         print(f"Original: {test}")
#         print(f"Cleaned:  {cleaned}")
#         print()

# test_deobfuscation()/

In [8]:
def aggregate_rationales(rationales, labels, post_length):
    """
    If all 3 annotators are normal → 3 zero spans → average (all zeros).
    If k annotators are non-normal and k spans exist → average the k spans (no added zeros).
    If k non-normal but fewer than k spans:
        If the missing annotators are non-normal → do not fill with zeros; average only existing spans and record rationale_support = #spans.
        If the missing annotators are normal (e.g., 2 hate + 1 normal + 2 spans) → append one zero span for the normal.
    """
    count_normal = labels.count(0)
    count_hate = labels.count(1)
    count_rationales = len(rationales)
    pad = np.zeros(post_length, dtype='int').tolist()

    # If there are hate labels but no rationales, something is wrong
    if count_hate > 0 and count_rationales == 0:
        return None

    # If all annotators are normal, return all zeros
    if count_normal == 3:
        return np.zeros(post_length).tolist()

    # If we have hate annotators
    if count_hate > 0:
        # Case 1: Number of rationales matches number of hate annotators
        if count_rationales == count_hate:
            return np.average(rationales, axis=0).tolist()

        # Case 2: Fewer rationales than hate annotators
        elif count_rationales < count_hate:
            # Add zero padding for normal annotators only
            rationales_copy = rationales.copy()
            zeros_to_add = count_normal
            for _ in range(zeros_to_add):
                rationales_copy.append(pad)
            return np.average(rationales_copy, axis=0).tolist()

        # Case 3: More rationales than hate annotators (shouldn't happen normally)
        else:
            # Just average what we have
            return np.average(rationales, axis=0).tolist()

    # Fallback: return zeros if no clear case matches
    return np.zeros(post_length).tolist()

In [9]:
import re
def process_raw_entries(data):
    """
    Process raw data entries
    """
    print("Processing raw entries...")
    processed_entries = {}
    dropped = 0

    for key, value in tqdm(data.items()):
        try:
            # Basic text processing
            raw_text = " ".join(value["post_tokens"])
            # Remove HTML tags <>
            raw_text = raw_text.replace("<", "").replace(">", "")
            # De-Obsfucate Patterns
            raw_text = deobfuscate_text(raw_text)
            # Remove punctuations
            raw_text = re.sub(r'[^\w\s]', '', raw_text)

            # Extract labels (1 = hate/offensive, 0 = normal)
            labels = []
            for annot in value["annotators"]:
                label = annot["label"]
                labels.append(1 if label in ['hatespeech', 'offensive'] else 0)

            # Process rationales
            rationales = value.get("rationales", [])
            aggregated_rationale = aggregate_rationales(rationales, labels, len(value["post_tokens"]))

            if aggregated_rationale is None:
                dropped += 1
                continue

            # Create entry
            entry = {
                "raw_text": raw_text,
                "hard_label": 1 if sum(labels) > len(labels) / 2 else 0,
                "soft_label": sum(labels) / len(labels),
                "rationales": aggregated_rationale
            }

            processed_entries[key] = entry

        except Exception as e:
            dropped += 1
            print(f"Error processing {key}: {e}")

    print(f"Processed: {len(processed_entries)}, Dropped: {dropped}")
    return processed_entries

In [10]:
# Demo, uncomment if needed
with open(data_path, 'r') as file:
  data = json.load(file)
entries = process_raw_entries(data)

Processing raw entries...


 90%|█████████ | 18203/20148 [00:04<00:00, 2445.99it/s]

Error processing 24439295_gab: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.


100%|██████████| 20148/20148 [00:05<00:00, 3407.25it/s]

Processed: 16538, Dropped: 3610


## Tokenizing and Rationale Alignment

In [11]:
def align_rationales(encoded, rationales, max_length=None, post_id=''):
    """
    Align rationales with tokenized text, handling subword tokenization.

    Args:
        encoded: Tokenized text from tokenizer
        rationales: List of rationale scores for original tokens
        max_length: Maximum sequence length (optional)

    Returns:
        Tensor of aligned rationale scores
    """
    word_ids = encoded.word_ids()
    aligned_rationales = []

    for word_id in word_ids:
        if word_id is None:  # Special tokens ([CLS], [SEP], [PAD])
            aligned_rationales.append(0.0)
        else:
            # Map back to original token rationale
            if word_id < len(rationales):
                aligned_rationales.append(float(rationales[word_id]))
            else:
                # Handle case where word_id exceeds rationales length
                print(f"Warning at {post_id}: word_id {word_id} exceeds rationales length ({len(rationales)}). Appending 0.0.")
                aligned_rationales.append(0.0)

    # Ensure we have the right length
    if max_length and len(aligned_rationales) > max_length:
        print(f"Warning at {post_id}: Truncating aligned rationales from {len(aligned_rationales)} to {max_length}.")
        aligned_rationales = aligned_rationales[:max_length]
    elif max_length and len(aligned_rationales) < max_length:
        print(f"Warning at {post_id}: Padding aligned rationales from {len(aligned_rationales)} to {max_length}.")
        aligned_rationales.extend([0.0] * (max_length - len(aligned_rationales)))

    return torch.tensor([aligned_rationales], dtype=torch.float)

In [12]:
# Align rationales using word_ids (available in most tokenizers)
def align_rationales(encoded, rationales, max_length=None, post_id=''):
    """
    Align rationales with tokenized text, handling subword tokenization.

    Args:
        encoded: Tokenized text from tokenizer
        rationales: List of rationale scores for original tokens
        max_length: Maximum sequence length (optional)

    Returns:
        Tensor of aligned rationale scores
    """
    word_ids = encoded.word_ids()
    aligned_rationales = []

    for word_id in word_ids:
        if word_id is None:  # Special tokens ([CLS], [SEP], [PAD])
            aligned_rationales.append(0.0)
        else:
            # Map back to original token rationale
            if word_id < len(rationales):
                aligned_rationales.append(float(rationales[word_id]))
            else:
                # Handle case where word_id exceeds rationales length
                print(f"Warning at {post_id}: word_id {word_id} exceeds rationales length ({len(rationales)}). Appending 0.0.")
                aligned_rationales.append(0.0)

    # Ensure we have the right length
    if max_length and len(aligned_rationales) > max_length:
        # print(f"Warning at {post_id}: Truncating aligned rationales from {len(aligned_rationales)} to {max_length}.")
        aligned_rationales = aligned_rationales[:max_length]
    elif max_length and len(aligned_rationales) < max_length:
        # print(f"Warning at {post_id}: Padding aligned rationales from {len(aligned_rationales)} to {max_length}.")
        aligned_rationales.extend([0.0] * (max_length - len(aligned_rationales)))

    return torch.tensor([aligned_rationales], dtype=torch.float)

def analyze_rationale_alignment(encoded, ori_rationales, aligned_rationales, tokenizer):
    """
    Analyze and print the alignment between original tokens and rationales.
    Useful for debugging and understanding the alignment process.
    """
    word_ids = encoded.word_ids()
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

    print("=== Rationale Alignment Analysis ===")
    print(f"Original rationales length: {len(ori_rationales)}")
    print(f"Tokenized sequence length: {len(tokens)}")
    print()
    # Original Rationale
    print("=== Original Rationale ===")
    current_word_id = None
    for i, (token, word_id) in enumerate(zip(tokens, word_ids)):
        if word_id != current_word_id:
            current_word_id = word_id
            if word_id is not None and word_id < len(ori_rationales):
                rationale_score = ori_rationales[word_id]
                if rationale_score > 0:
                    print(f"Word {word_id}: '{token}' -> Rationale: {rationale_score}")
        elif word_id is not None:
            # This is a subword token
            if word_id < len(ori_rationales) and ori_rationales[word_id] > 0:
                print(f"  Subword: '{token}' -> Rationale: {ori_rationales[word_id]}")
    print()

    print("=== Aligned Rationale ===")
    # Aligned rationale (length SHOULD be the same if done correctly)
    current_word_id = None
    for i, (token, word_id) in enumerate(zip(tokens, word_ids)):
      if aligned_rationales[i] != 0:
        print(f"Word {word_id}: '{token}' -> Rationale: {aligned_rationales[i]}")


In [13]:
# # Example run, uncomment this for demo
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# for key, sample in entries.items():
#   encoded = tokenizer(sample['raw_text'], max_length=40,
#                       padding='max_length', truncation=True,
#                       return_tensors='pt')
#   aligned_rationales = align_rationales(encoded, sample['rationales'], max_length=50, post_id=key)
#   # analyze_rationale_alignment(encoded, sample['rationales'], aligned_rationales[0], tokenizer)

In [ ]:
# e = tokenizer(entries['1178929672379387904_twitter']['raw_text'])
# print(entries['1178929672379387904_twitter']['raw_text'])
# print(tokenizer.decode(e['input_ids']))
# print(tokenizer.convert_ids_to_tokens(e['input_ids']))
# print(e.word_ids())


she was also the best player and second top goalscorer with the super falcons who won the number number number african women championship october i st making nigeria proud sunday dare nigeria independence
[CLS] she was also the best player and second top goalscorer with the super falcons who won the number number number african women championship october i st making nigeria proud sunday dare nigeria independence [SEP]
['[CLS]', 'she', 'was', 'also', 'the', 'best', 'player', 'and', 'second', 'top', 'goalscorer', 'with', 'the', 'super', 'falcons', 'who', 'won', 'the', 'number', 'number', 'number', 'african', 'women', 'championship', 'october', 'i', 'st', 'making', 'nigeria', 'proud', 'sunday', 'dare', 'nigeria', 'independence', '[SEP]']
[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, None]


## Prepare Dataset

In [14]:
from typing import List, Tuple
from torch.utils.data import Dataset

class HateXplainDataset(Dataset):
  def __init__(
    self,
    features:List[Tuple[str, List[float]]],
    labels: List[int],
    model_name: str = 'distilbert-base-uncased',
    max_length: int = 128,
    ):
      self.texts = [feat[0] for feat in features]
      self.rationales = [feat[1] for feat in features]
      self.labels = labels
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.max_length = max_length

  def __len__(self):
    return len(self.labels)

  def _align_rationales(self, encoded, rationales, max_length=None):
    """
    Align rationales with tokenized text, handling subword tokenization.

    Args:
        encoded: Tokenized text from tokenizer
        rationales: List of rationale scores for original tokens
        max_length: Maximum sequence length (optional)

    Returns:
        Tensor of aligned rationale scores
    """
    word_ids = encoded.word_ids()
    aligned_rationales = []

    for word_id in word_ids:
        if word_id is None:  # Special tokens ([CLS], [SEP], [PAD])
            aligned_rationales.append(0.0)
        else:
            # Map back to original token rationale
            if word_id < len(rationales):
                aligned_rationales.append(float(rationales[word_id]))
            else:
                # Handle case where word_id exceeds rationales length
                print(f"Warning: word_id {word_id} exceeds rationales length ({len(rationales)}). Appending 0.0.")
                # print(f"Text {self.tokenizer.decode(encoded['input_ids'][0])}")
                aligned_rationales.append(0.0)

    # Ensure we have the right length
    if max_length and len(aligned_rationales) > max_length:
        # print(f"Warning: Truncating aligned rationales from {len(aligned_rationales)} to {max_length}.")
        aligned_rationales = aligned_rationales[:max_length]
    elif max_length and len(aligned_rationales) < max_length:
        # print(f"Warning: Padding aligned rationales from {len(aligned_rationales)} to {max_length}.")
        aligned_rationales.extend([0.0] * (max_length - len(aligned_rationales)))

    return torch.tensor([aligned_rationales], dtype=torch.float)

  def __getitem__(self, idx):
    encoding = self.tokenizer(
        self.texts[idx],
        max_length = self.max_length,
        padding='max_length', truncation=True,
        return_tensors='pt'
      )
    rationales = self._align_rationales(encoding, self.rationales[idx], self.max_length)

    return {
        'input_ids': encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'rationales': rationales.flatten(),
        'labels': torch.tensor(self.labels[idx], dtype=torch.long) # Changed dtype to torch.long
    }

In [ ]:
# Demo, uncomment if needed
# texts = [entry['raw_text'] for entry in entries]
# rationales = [entry['rationales'] for entry in entries]
# labels = [entry['hard_label'] for entry in entries] #use hard label for now
# model_name = 'distilbert-base-uncased'
# max_length = 128

# dataset = HateXplainDataset(texts, rationales, labels, model_name, max_length)
# from torch.utils.data import DataLoader

# loader = DataLoader(dataset, batch_size=32, shuffle=True)
# sample = next(iter(loader))
# # Dataset Overview
# print("=== Dataset Overview ===")
# print(f"Number of samples: {len(dataset)}")
# print(f"Number of batches: {len(loader)}")
# print()
# print(f"Sample keys: {sample.keys()}")
# print(f"Sample input_ids shape: {sample['input_ids'].shape}")
# print(f"Sample attention_mask shape: {sample['attention_mask'].shape}")
# print(f"Sample rationales shape: {sample['rationales'].shape}")
# print(f"Sample labels shape: {sample['labels'].shape}")

# Modelling

In [15]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoConfig

class HateClassifier(nn.Module):
  def __init__(self, model_name, num_classes, **kwargs):
    super(HateClassifier, self).__init__()
    self.config = AutoConfig.from_pretrained(model_name, **kwargs)
    self.config.num_labels = num_classes
    self.classifier = AutoModelForSequenceClassification.from_pretrained(model_name, config=self.config)

  def forward(self, input_ids, attention_mask):
    outputs = self.classifier(input_ids=input_ids, attention_mask=attention_mask)
    return outputs

# Pipeline
The whole ass training and evaluation process here. Customize here if needed

## Prepare Data

In [16]:
# Prepare Data
from sklearn.model_selection import train_test_split
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

with open(data_path, 'r') as f:
    data = json.load(f)

In [17]:
processed_data = process_raw_entries(data)

Processing raw entries...


 92%|█████████▏| 18625/20148 [00:04<00:00, 3761.51it/s]

Error processing 24439295_gab: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.


100%|██████████| 20148/20148 [00:05<00:00, 3977.89it/s]

Processed: 16538, Dropped: 3610


In [18]:
print("Sample processed data:")
for key, value in processed_data['25316209_gab'].items():
  print(f"{key}: {value}")

Sample processed data:
raw_text: what was that english lol that was one too many negatives in a sentence no i did not say you specifically lard ass i said women in general learn to read cleaely i can and you can not
hard_label: 1
soft_label: 0.6666666666666666
rationales: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [35]:
with open('/content/drive/MyDrive/Thesis/data/post_id_divisions.json') as file:
  post_id_divisions = json.load(file)

# Train
X_train = []
y_train = []
train_missing = 0
for train_key in post_id_divisions['train']:
  try:
    X_train.append((processed_data[train_key]['raw_text'], processed_data[train_key]['rationales']))
    y_train.append(processed_data[train_key]['hard_label'])
  except Exception as e:
    train_missing += 1
print(f"Train missing: {train_missing}")

# Val
X_val = []
y_val = []
val_missing = 0
for val_key in post_id_divisions['val']:
  try:
    X_val.append((processed_data[val_key]['raw_text'], processed_data[val_key]['rationales']))
    y_val.append(processed_data[val_key]['hard_label'])
  except Exception as e:
    val_missing += 1
print(f"Val missing: {val_missing}")

# Test
X_test = []
y_test = []
test_missing = 0
for test_key in post_id_divisions['test']:
  try:
    X_test.append((processed_data[test_key]['raw_text'], processed_data[test_key]['rationales']))
    y_test.append(processed_data[test_key]['hard_label'])
  except Exception as e:
    test_missing += 1
print(f"Test missing: {test_missing}")

Train missing: 2156
Val missing: 257
Test missing: 278


In [49]:
Counter(y_train), Counter(y_val), Counter(y_test)

(Counter({1: 9131, 0: 4096}),
 Counter({0: 524, 1: 1141}),
 Counter({0: 504, 1: 1142}))

In [37]:
len(X_train), len(X_val), len(X_test)

(13227, 1665, 1646)

In [38]:
# Import this from config later
model_name = 'distilbert-base-uncased'
max_length = 128

train_dataset = HateXplainDataset(
    features=X_train,
    labels=y_train,
    model_name=model_name,
    max_length=max_length
)

val_dataset = HateXplainDataset(
    features=X_val,
    labels=y_val,
    model_name=model_name,
    max_length=max_length
)

test_dataset = HateXplainDataset(
    features=X_test,
    labels=y_test,
    model_name=model_name,
    max_length=max_length
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [39]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) # Use shuffle=False for validation
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # Use shuffle=False for testing

In [40]:
model = HateClassifier(model_name, num_classes=2)
# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device {device}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device cuda


In [41]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

In [42]:
from tqdm import tqdm

def train_epoch(loader, optimizer, criterion):
  """Train for one epoch"""
  model.train()
  total_loss = 0
  correct_predictions = 0
  total_predictions = 0
  for batch in tqdm(loader, desc="Training"):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()
    outputs = model(input_ids, attention_mask)
    loss = criterion(outputs.logits, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    _, predicted = torch.max(outputs.logits, 1)
    total_predictions += labels.size(0)
    correct_predictions += (predicted == labels).sum().item()

  avg_loss = total_loss / len(loader)
  accuracy = correct_predictions / total_predictions
  return avg_loss, accuracy

In [43]:
def evaluate(loader, criterion):
  """Eval on val set"""
  model.eval()
  total_loss = 0
  correct_predictions = 0
  total_predictions = 0

  with torch.no_grad():
    for batch in tqdm(loader, desc="Evaluating"):
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      outputs = model(input_ids, attention_mask)
      loss = criterion(outputs.logits, labels)

      total_loss += loss.item()
      _, predicted = torch.max(outputs.logits, 1)
      total_predictions += labels.size(0)
      correct_predictions += (predicted == labels).sum().item()

  avg_loss = total_loss / len(loader)
  accuracy = correct_predictions / total_predictions
  return avg_loss, accuracy

In [44]:
# Training loop
epochs = 2
for epoch in range(epochs):
  # Train
  train_loss, train_acc = train_epoch(train_loader, optimizer, criterion)

  # Validate
  val_loss, val_acc = evaluate(test_loader, criterion)

  print(f"Epoch {epoch+1}/{epochs}")
  print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
  print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

Training:  30%|██▉       | 123/414 [00:37<01:26,  3.35it/s]

Training:  41%|████      | 168/414 [00:50<01:13,  3.34it/s]

Training:  59%|█████▊    | 243/414 [01:12<00:50,  3.36it/s]

Training:  74%|███████▍  | 306/414 [01:32<00:32,  3.29it/s]

Training:  78%|███████▊  | 324/414 [01:37<00:27,  3.33it/s]

...

Training:  63%|██████▎   | 260/414 [01:20<00:47,  3.21it/s]

Training:  81%|████████  | 334/414 [01:43<00:25,  3.19it/s]

Evaluating:  52%|█████▏    | 27/52 [00:03<00:02,  8.88it/s]

Evaluating: 100%|██████████| 52/52 [00:05<00:00,  9.14it/s]

Epoch 2/2
Train Loss: 0.3359, Train Acc: 0.8526
Val Loss: 0.3602, Val Acc: 0.8372


# Task
Infer on the test dataset using the trained model, output the predictions to a JSON file, and print the F1 score and accuracy in the notebook.

## Define inference function

### Subtask:
Create a function to perform inference on a given data loader using the trained model.


**Reasoning**:
Define the inference function as instructed, which will take a data loader, model, and device as input and return the predictions and true labels.



In [45]:
def inference(loader, model, device):
    """
    Perform inference on a given data loader.

    Args:
        loader: PyTorch DataLoader
        model: Trained PyTorch model
        device: Device to run inference on (e.g., 'cuda', 'cpu')

    Returns:
        Tuple of lists: (all_predictions, all_labels)
    """
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs.logits, 1)

            all_predictions.extend(predicted.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    return all_predictions, all_labels

## Perform inference

### Subtask:
Use the defined inference function to get predictions for the test dataset.


**Reasoning**:
Use the defined inference function to get predictions for the test dataset.



In [46]:
predictions, true_labels = inference(test_loader, model, device)

Inference:  52%|█████▏    | 27/52 [00:02<00:02,  9.24it/s]

Inference: 100%|██████████| 52/52 [00:05<00:00,  9.27it/s]


In [47]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")

Accuracy: 0.8371810449574727
F1 Score: 0.8849785407725322
